# 1 · Your first Markov chain

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/EN/jupyter/markov-book-en/01_first_markov_chain.ipynb)

A **Markov chain** builds a sequence one step at a time. At every step it looks only at the last
few symbols (its *memory*) and picks the next symbol with certain probabilities.

In this chapter there is no music yet. We use actions and letters, so you can see how little is
needed, and meet the three GreasyPidgin objects we will use all day:

- `MarkovRule` — one row of the recipe: *"after this, one of those comes next, with these weights"*
- `MarkovRules` — a collection of rules
- `MarkovChain` — follows the rules and produces a sequence

Run the cells from top to bottom with **Shift + Enter**. The first cell installs everything.

In [ ]:
# Setup -- runs by itself (about 20 seconds the first time)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # which version of GreasyPidgin to use (a git branch or tag)
TUTORIAL_REF  = "main"   # which version of this tutorial's helper file to use
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/EN/jupyter/markov-book-en/workshop_helpers.py",
        "workshop_helpers.py")

TEXT_FILE = "Schwitters-Ursonate-score.txt"   # text for this chapter (lives in the MATERIAL folder)
if not pathlib.Path(TEXT_FILE).exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/MATERIAL/{TEXT_FILE}", TEXT_FILE)

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

print("Ready!")

## A chain you write by hand: homage to Cage's *Song Books*

The states of a chain do not have to be notes; they can be **actions**.

Each rule says what can follow a state, and how likely each option is.
The weights are normalised automatically, so `[6, 3, 1]` works just as well as `[0.6, 0.3, 0.1]`.
First pure chance: every row has the same weights, so it does not matter what just happened.

In [ ]:
actions = ["sing", "speak", "pause", "noise", "action"]

# Every row has the same weights (in the order of `actions`): pure chance
chance = MarkovChain(rules=MarkovRules([
    MarkovRule(a, actions, [3, 3, 2, 2, 1]) for a in actions
]))

Now let the chain run for 12 steps. `seed` is *where the chain starts* (not a random seed),
`n` is how many new steps to take. Run the cell several times: every run gives a different sequence.

In [ ]:
print(" | ".join(chance.getNElements(seed="pause", n=12)))

Now the same actions **with memory**: what just happened changes the odds for what comes next.
That is the difference between a chance operation and a Markov chain.
The weights are again in the order `sing, speak, pause, noise, action`.

In [ ]:
memory = MarkovChain(rules=MarkovRules([
    MarkovRule("sing",   actions, [6, 1, 1, 1, 1]),   # singing tends to continue
    MarkovRule("speak",  actions, [1, 4, 2, 1, 2]),
    MarkovRule("pause",  actions, [2, 2, 1, 1, 4]),   # after silence something often happens
    MarkovRule("noise",  actions, [1, 1, 3, 4, 1]),   # a noise stays or dies away
    MarkovRule("action", actions, [1, 3, 4, 1, 1]),   # after an action, calm returns
]))
print(memory.rules)
print(" | ".join(memory.getNElements(seed="pause", n=12)))

### Seeing the rules

A table of probabilities is easier to read as a picture. Each row is the *current* state,
each column the *next* one; the brighter, the more likely.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
wh.plot_transition_matrix(chance, label=str, ax=axes[0])
wh.plot_transition_matrix(memory, label=str, ax=axes[1])
axes[0].set_title("pure chance")
axes[1].set_title("with memory")
plt.tight_layout()
plt.show()

**Try it:** change the weights in `memory` so that a pause is almost always followed by an action.
Then perform the sequence as a score: one person reads it out, the group sings, speaks, stays silent,
and makes noises and actions.

## Learning a chain from data

Writing rules by hand gets tiring. A chain can also **learn its rules from an existing sequence**
by counting what followed what. Here the sequence is the text of Kurt Schwitters' *Ursonate* (1922–32),
one letter per step.

In [ ]:
with open(TEXT_FILE, encoding="utf-8") as f:
    letters = list(f.read())        # one letter (or space) per step
print(len(letters), "characters")

learned = MarkovChain()
learned.generateRulesFromList(letters, order=1)   # order 1: remember only the last letter
print(learned.rules["r"])                          # what follows an "r"?

In [ ]:
new = learned.getNElements(seed=letters[0], n=200)
print("".join(new))

### Seeing the learned rules

These rules can be looked at as a picture, too. Each row is the *current* letter, each column the *next* one
(`␣` is the space); the brighter, the more likely.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
wh.plot_transition_matrix(learned, label=lambda s: "␣" if s == " " else s, ax=ax, max_states=100)
ax.tick_params(labelsize=7)
plt.show()

The chain only knows which letter follows which, and how often. That is all — so it invents syllables and words
with the same "taste" as the original.

### Memory: order 2

With `order=2` the chain remembers the last **two** letters. The seed must then have two letters, too.

In [ ]:
order = 2
learnedN = MarkovChain()
learnedN.generateRulesFromList(letters, order=order)

print("".join(learnedN.getNElements(seed=tuple(letters[:order]), n=200)))

**Try it:**

- Replace `TEXT_FILE` with a text of your own, e.g. `TEXT_FILE = "my_text.txt"` (in Colab: upload the file in the
  file browser on the left), and run the cells from the reading step again.
- Compare `order=1`, `2`, `3`, `5`. What happens to the output when the order gets large?
  (Hint: with a lot of memory, the chain has only one choice at every step —
  it can only copy the original.)

That trade-off between *copying* and *inventing* is the main idea of this workshop.
In the next chapter we do the same with notes from a MIDI file.